之前我们了解了checkpoints（检查点）的功能。Checkpoints不仅仅是让我们可以保存Graph执行过程中的每一步状态，更重要的是它实现了LangGraph的持久化层（persistence layer）。
当我们在编译Graph时引入checkpointer，它会在每个super-step自动保存Graph状态的快照。这些checkpoints被保存在Thread（线程）中，使得我们可以在Graph执行完成后仍然访问Graph的状态。
正是因为这个持久化层，很多强大的功能才成为可能，包括Human-in-the-loop的人机交互、记忆（Memory）、时间旅行（Time Travel）和容错（Fault-tolerance）等。
基于持久化层有一个强大的功能--Interrupts（中断），它允许我们在Graph执行的特定位置暂停，等待外部输入后再继续执行，这为实现Human-in-the-loop模式提供了基础。
### 1.Interrupts功能概述
有一些AI编程助手，当AI需要执行某些关键操作（比如修改当前文件、允许可能影响系统的命令）时，会暂停并询问你是否确认呢，只有在你批准后才能继续执行。
在日常生活中，类似的情景非常普遍：
- 在执行关键操作（比如发送邮件、转账、操作数据）之前，需要人工审批
- 在LLM生成内容后，需要人工审查和编辑
- 在执行工具调用前，需要人工确认和修改参数
- 需要收集用户输入，并在输入无效时重新询问
为了解决这些问题，LangGraph提供了Interrupts，它允许Graph在执行过程中暂停，等待外部输入（比如人工审批、用户确认呢等），然后根据输入决定如何继续执行。
Interrupts的核心价值不仅仅在于暂停Graph的执行，更重要的是它实现了对Graph State的高级控制：
- 暂停执行：在Graph执行的任何位置暂停，保存当前State的完整快照到持久层
- 接收外部输入：在暂停期间，可以接收来自外部的输入（人工审批、用户输入、系统决策等）
- 基于输入继续执行：外部输入会被传递回Graph，成为interrupt()调用的返回值，Graph可以基于这个输入值来决定后续的执行路径和State的更新方式
- 动态控制State：通过外部输入，我们可以修改State的值、改变执行路由、甚至重新定义Graph的行为，这是对State的一种高级控制能力
这种能力让我们能够实现真正的Human-in-the-loop：Graph可以在需要人类决策的关键节点暂停，等待人类的输入，然后根据人类的决策继续执行。这不仅让Graph的执行流程更加灵活可控，也让State的管理更加精细和动态。
### 2.Interrupts的实现方法
#### 2.1.第一步：使用interrupt()暂停执行
首先，需要使用interrupt函数来暂停Graph的执行。当你在节点中调用interrupt时，LangGraph会保存当前的state，然后等待你提供一个输入来恢复执行。
示例：
```python
from langgraph.types import interrupt

def approval_node(state: State):
    """审批节点"""
    # 暂停并请求审批
    approved = interrupt("Do you approve this action?")

    # 当你恢复时，Command(resume=...)的值会在这里返回
    return {"approved": approved}
```
当Graph执行到第一行approved = interrupt("Do you approve this action?")时，会发生以下事情：
1. 执行立即暂停：Graph的执行会立即暂停在这一行，不会继续往下执行。即approved变量此时还没有被赋值，代码就停在这里了。
2. 传递payload：interrupt()函数括号中的参数（这里是Do you approve this action?）被成为payload（载荷）。在interrupt()函数中，payload就是你想要传递给Graph调用者的实际信息内容。这个payload可以是任何JSON可序列化的值，比如字符串、数字、对象、数组等。
这个payload的作用是什么呢？当Graph暂停时，这个payload会返回到Graph的调用者，出现在返回结果的__interrupt__字段中。这样调用者就知道Graph在等待什么，可以根据这个信息来决定如何恢复执行。
3. 保存当前状态：暂停时，LangGraph会通过预先设定好的checkpointer将当前的Graph状态保存到持久化层。这个状态包含了State的值，当前执行到哪个节点、执行位置等完整信息。这个状态会被保存在指定的Thread中（通过thread_id表示），这样即使程序重启，也能恢复执行。
4. 等待恢复：Graph会一直等待，直到你通过Command(resume=...)恢复执行。由于状态已经保存到持久化层，即使等待很长时间或者程序重启，也能恢复执行。
当执行：return {"approved": approved}时：
一般情况下，LangGraph中节点函数返回的字典（如：{"approved": approved}）将更新Graph的State。这个字典中的键（这里是"approved"）对应已在State中预先定义的字段名，值（这里是approved变量的值）就是要更新到State中的新值。
但是，由于interrupt的存在，Graph的执行已经在上一段代码处就暂停了，而本行代码在Graph暂停时同样不会执行。只有当Graph恢复执行后，这一行才会执行。
那么，approved这个变量的值到底是如何变化呢？这就涉及到interrupt()函数的返回值机制了：
- 当Graph暂停时：interrupt()函数不会立即返回，而是暂停执行，所以approved变量此时还没有被赋值
- 当Graph恢复时：interrupt()函数会返回用户通过Command(resume=...)传递的值。比如，如果你恢复时传入Command(resume=True)，那么interrupt()就会返回True，这个True值就会被赋值给approved变量。
需要注意的是，如前所述，interrupt函数的运作涉及到state的保存和恢复，故在使用之前需要设置好两个东西：
1. checkpointer：用于保存当前的state，即checkpoints，保存在指定的thread中。
2. thread id：在config中设置thread_id，这样runtime才知道要把状态保存到哪个Thread，以及从哪个Thread恢复状态。
#### 2.2.第二步：使用Command(resume=...)恢复中断
当Graph因为interrupt()而暂停后，我们需要通过再次调用Graph来恢复执行。
```python
from langgraph.types import Command

# 初始允许 - 遇到interrupt并暂停
# thread_id是持久化指针（生产环境中应存储稳定的ID）
config = {"configurable": {"thread_id": "thread-1"}}
result = grpah.invoke({"input": "data"}, config=config)

# 检查被中断的内容
# __interrupt__包含传递给interrupt()的payload
print(result["__interrupt__"])
# >[Interrupt(value="Do you approve this action?")]

# 使用人工响应恢复
# resume payload会成为节点内interrupt()的返回值
graph.invoke(Command(resume=True), config=config)
```
首先，这里设置了config，其中thread_id是关键。恢复执行时必须使用与中断发生时相同的thread_id，这样LangGraph才能从持久化层中找到之前保存的状态。thread_id就像是持久化游标，重复使用同一个thread_id会恢复同一个Thread的状态。
```python
result = grpah.invoke({"input": "data"}, config=config)

# 检查被中断的内容
# __interrupt__包含传递给interrupt()的payload
print(result["__interrupt__"])
# >[Interrupt(value="Do you approve this action?")]
```
这是首次执行Graph。结合第一步的approval_node函数，当Graph执行到approval_node节点时，会执行函数中的approved=interrupt("Do you approve this action?")这一行。当执行到interrupt()函数时，Graph会立即暂停，不会继续执行后面的return {"approved": approved}代码。
这时，返回的result会包含Graph暂停时的状态信息。如果打印result["__interrupt__"]字段，可以看到它包含了传递给interrupt()的payload（这里是“Do you approve this action?”）。打印的结果就是[Interrupt(value="Do you approve this action?")]，这样我们就会知道Graph在等待什么，可以根据这个信息来决定如何恢复执行。
```python
graph.invoke(Command(resume=True), config=config)
```
这是恢复Graph执行的关键步骤。Command(resume=True)中的True值会被传递回节点中interrupt()的调用位置，成为interrupt()函数的返回值。
当执行这一行时，会发生以下事情：
1. 加载保存的状态：LangGraph会通过checkpointer从持久化层加载之前保存的状态（使用相同的thread_id）。这个状态包含了Graph暂停时的完整信息，包括State的值、当前执行到approval_node节点、以及执行位置（即approval_node节点中的approved=interrupt("Do you approve this action?")这一行）。
2. 恢复节点执行：Graph会从approval_node节点开始重新执行。注意，节点会重新开始执行，也就是说，approval_node函数会从头开始运行。如果函数中有interrupt()之前的代码，这些代码会再次执行。
3. 传递resume值：当执行到approved=interrupt("Do you approve this action?")这一行时，interrupt()函数会返回你通过Command(resume=...)传递的值。在这个例子中，Command(resume=True)传递的True值就会被赋值给approved变量。
4. 继续执行：节点继续执行interrupt()之后的代码，也就是return {"approved": approved}。此时approved的值是True，所以返回{"approved": True}。LangGraph会将这个返回值更新到State中，State的approved字段被设置为True，节点执行完成。
Command(resume=...)中的值可以是任何JSON可序列化的值，比如：简单的布尔值、字符串、数字、对象、数组等。
#### 2.3.补充：在实际应用中处理中断
在实际的工程应用中，两次的graph.invoke()调用通常发生在应用程序层（比如Web应用、API服务、CLI工具等），而不是直接写在Graph的脚本中。
基本的处理流程示例：
```python
from langgraph.types import Command

def handle_graph_execution(user_input: str, thread_id: str):
    """应用程序的主函数，处理Graph执行和中断"""
    config = {"configurable": {"thread_id": thread_id}}
    # 第一次调用：启动Graph执行
    # 当Graph执行到interrupt()时会暂停，返回结果
    result = graph.invoke({"input": user_input}, config=config)

    # 循环处理所有中断（Graph可能多次中断）
    while "__interrupt__" in result and result["__interrupt__"]:
        # 提取中断信息
        # 获取的就是interrupt()函数中的payload
        interrupt_data = result["__interrupt__"][0].value
        # 显示UI给用户（Web界面、CLI提示等）
        print(f"需要审批：{interrupt_data}")
        # 获取用户响应（Web表单、CLI输入等）
        user_decision = input("请输入决定：")

        # 第二次调用：恢复Graph执行
        # 关键：必须使用相同的thread_id
        result = graph.invoke(Command(resume=user_decision), config=config)

    # Graph执行完成
    return result
```
### 3.Interrupts的常见应用模式示例
Interrupts的核心功能是暂停执行等待外部输入。常见使用场景：
- 审批工作流：在执行关键操作前暂停（API调用、数据库更改、金融交易）
- 审查和编辑：让人类在继续之前审查和修改LLM输出或工具调用
- 中断工具调用：在执行工具调用前暂停，以便在执行前审查和编辑工具调用
- 验证人工输入：在继续下一步之前暂停，以验证人工输入

具体使用场景可参考：
https://docs.langchain.com/oss/python/langgraph/interrupts#stream-with-human-in-the-loop-hitl-interrupts